# Qwen3-4B Thinking: pythoncodes SFT + 5 CL LoRA

Параметры обучения лежат в `configs/train/lora_pythoncodes_cl.yaml`. Ноутбук только запускает и собирает таблицы.


In [ ]:
import os, sys
from pathlib import Path

os.environ['SELECTED_MODEL'] = '4b-thinking'
os.environ['MAX_TOKENS'] = '4096'

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import src.config as config

print(config.MODEL_NAME)
print(config.MODEL_PATH)

## Скачать модель, если локально её ещё нет


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id=config.MODEL_NAME,
    local_dir=config.MODEL_PATH,
    local_dir_use_symlinks=False,
    resume_download=True,
)


## Проверить датасет и список запусков


In [ ]:
from src.train.lora_train import load_train_config, load_pythoncodes_cl_df

train_cfg = load_train_config()
df = load_pythoncodes_cl_df(train_cfg)

display(df.head(3))
print(df.shape)

cat_cols = [c for c in df.columns if c.endswith('_category')]
print(cat_cols)

for c in cat_cols:
    print('\n' + c)
    print(df[c].value_counts(dropna=False))

display(pd.DataFrame(train_cfg['runs']))


## Smoke-run


In [ ]:
from src.train.lora_train import train_lora_experiments, collect_training_table

train_cfg = load_train_config()
train_cfg['dataset']['limit'] = 2000
train_cfg['dataset']['val_size'] = 100
train_cfg['training']['max_steps'] = 10
train_cfg['training']['stage_max_steps'] = 10
train_cfg['training']['eval_steps'] = 10
train_cfg['training']['save_steps'] = 10
train_cfg['training']['report_to'] = 'none'
smoke = train_lora_experiments(train_cfg=train_cfg, runs=train_cfg['runs'][:1])
display(smoke)


## Обучение


In [ ]:
# Все 6 запусков:
# train_cfg = load_train_config()
# train_df = train_lora_experiments(train_cfg=train_cfg)
# display(train_df)

# Один запуск:
train_df = train_lora_experiments(only={'cl_length'})
display(train_df)

# Продолжить с третьего запуска:
# train_df = train_lora_experiments(skip=2)
# display(train_df)


## Таблица логов обучения


In [ ]:
train_table = collect_training_table()
display(train_table)
print(config.OUTPUTS_DIR / 'train_runs' / 'summary.csv')
print(config.OUTPUTS_DIR / 'train_runs' / 'summary.parquet')


## Eval/inference в pandas


In [ ]:
from src.train.lora_train import prepare_lora_experiments, evaluate_lora_experiments

experiments = prepare_lora_experiments(include_base=True)
display(pd.DataFrame(experiments, columns=['experiment', 'model_path', 'adapter_path']))

# Smoke eval:
eval_df = evaluate_lora_experiments(experiments, max_tasks=5, out_name='cl_eval_smoke')
display(eval_df)

# Full eval:
# eval_df = evaluate_lora_experiments(experiments, out_name='cl_eval_full')
# display(eval_df)
